# Crash Course — 보행분석

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/gait-analysis.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/gait-analysis.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 표준 분석 흐름

실제 파일이 필요한 단계의 명령 형태(실행 예가 아니라 모양만):

In [ ]:
import pandas as pd
# gaitpdb 신호 파일: 머리글 없음, 탭 구분, 19열(1열 시간, 18·19열 = 왼발·오른발 총 힘, N)
df = pd.read_csv("data/gaitpdb/<파일 이름>_01.txt", sep="\t", header=None)
left_total = df[17].to_numpy()       # 0부터 세면 17번이 18열

## 최소 실행 예시

### 예시 1 — 힘 threshold로 stride를 잘라 참가자 단위 CV 구하기

가상 참가자 두 명의 한쪽 발 수직력(100 Hz)을 만들고 20 N threshold로 접지와 떨어짐을 찾는다. 두 번째 참가자 자료에는 swing 중 잡음 spike 하나가 threshold를 넘었다. "최소 접지 시간 100 ms" 규칙이 있을 때와 없을 때 CV를 비교한다.

In [ ]:
import numpy as np
import pandas as pd

fs = 100                                     # gaitpdb와 같은 100 Hz
rng = np.random.default_rng(7)

def one_foot_vgrf(stride_times, stance_frac=0.62, bw=700):
    """stride마다 길이가 다른 한쪽 발 수직력(N) 가상 신호: stance 동안 반원 모양 + 잡음."""
    out = []
    for T in stride_times:
        n, ns = int(round(T * fs)), int(round(T * stance_frac * fs))
        x = np.zeros(n)
        x[:ns] = 1.1 * bw * np.sin(np.pi * np.arange(ns) / ns)
        out.append(x)
    return np.concatenate(out) + rng.normal(0, 5, sum(len(o) for o in out))

def stride_table(f, thr=20, min_contact_s=0.0):
    on = np.flatnonzero((f[1:] >= thr) & (f[:-1] < thr)) + 1    # 접지(위로 교차)
    off = np.flatnonzero((f[1:] < thr) & (f[:-1] >= thr)) + 1   # 떨어짐(아래로 교차)
    if min_contact_s > 0:                                       # 짧은 "접지"(잡음 spike)는 버린다
        keep = [i for i, a in enumerate(on)
                if (off[off > a][:1] - a).size and (off[off > a][0] - a) / fs >= min_contact_s]
        on = on[keep]
    rows = []
    for a, b in zip(on[:-1], on[1:]):                           # 접지–다음 접지 = 1 stride
        o = off[(off > a) & (off < b)]
        if len(o) == 1:
            rows.append({"stride_s": (b - a) / fs, "stance_pct": 100 * (o[0] - a) / (b - a)})
    return pd.DataFrame(rows).iloc[1:-1]                        # 첫·마지막 stride는 버린다

people = {"P01": rng.normal(1.10, 0.015, 40), "P02": rng.normal(1.05, 0.045, 40)}
summary = []
for pid, st in people.items():
    f = one_foot_vgrf(st)
    for rule in [0.0, 0.1]:                                     # 규칙 없음 vs 최소 접지 100 ms
        tab = stride_table(f, min_contact_s=rule)
        summary.append({"참가자": pid, "최소 접지(s)": rule, "stride 수": len(tab),
                        "stride time 평균(s)": round(tab.stride_s.mean(), 3),
                        "stride time CV(%)": round(100 * tab.stride_s.std(ddof=1) / tab.stride_s.mean(), 2),
                        "stance(%)": round(tab.stance_pct.mean(), 1)})
print(pd.DataFrame(summary).to_string(index=False))

### 예시 2 — freeze index를 window마다 계산하기

64 Hz 가상 shank 가속도(mg)에 0–20초 걷기, 20–25초 동결(6 Hz 떨림), 25–30초 정지를 넣고, 4초 window를 밀며 FI를 계산한다. 두 대역 에너지의 합이 작으면(정지) 판정하지 않는다.

In [ ]:
import numpy as np
from scipy import signal

fs = 64                                       # Bächlin 2010과 같은 64 Hz
rng = np.random.default_rng(3)
t = np.arange(0, 30, 1 / fs)
x = (300 * np.sin(2 * np.pi * 1.0 * t)         # 0–20 s: 1 Hz 걸음 리듬(mg)
     + 90 * np.sin(2 * np.pi * 2.0 * t)        #         + 2·4 Hz harmonic
     + 40 * np.sin(2 * np.pi * 4.0 * t))
fog = (t >= 20) & (t < 25)                     # 20–25 s: 가상 동결(6 Hz 떨림 + 작은 1.5 Hz 움직임)
x[fog] = 80 * np.sin(2 * np.pi * 6.0 * t[fog]) + 60 * np.sin(2 * np.pi * 1.5 * t[fog])
x[t >= 25] = 0                                 # 25–30 s: 정지
x += rng.normal(0, 5, t.size)                  # 센서 잡음

win, step = 4 * fs, fs // 2                    # 4 s window(256 표본), 0.5 s씩 이동
for start in range(0, t.size - win + 1, 4 * step):          # 출력은 2 s 간격만
    seg = x[start:start + win]
    f, p = signal.periodogram(seg - seg.mean(), fs=fs, window="boxcar")
    loco = p[(f >= 0.5) & (f < 3)].sum()
    frz = p[(f >= 3) & (f <= 8)].sum()
    gate = (loco + frz) > 1e3                  # 에너지가 작으면(정지) 판정하지 않음
    fi = frz / loco if gate else float("nan")
    print(f"{t[start]:4.1f}–{t[start] + 4:4.1f} s  FI = {fi:8.2f}  {'(에너지 낮음: 판정 안 함)' if not gate else ''}")